# 统一追踪系统（人脸 + 颜色）

支持键盘快捷键实时切换追踪模式，舵机控制使用平滑方案。

### 快捷键
| 按键 | 功能 |
|------|------|
| **F** | 人脸追踪模式 |
| **C** | 颜色追踪模式 |
| **1/2/3/4** | 选择颜色（红/绿/蓝/黄）并切换到颜色追踪 |
| **G** | 取消追踪（待机） |
| **Q / ESC** | 退出程序 |

执行顺序：依次运行下面所有单元。

## 1. 导入包与集中配置

所有可调参数集中在此单元。人脸追踪与人脸追踪共用先进的"时间基速度 PID + 低通滤波 + 加速度限幅 + 跳帧发送"平滑方案。

In [9]:
import time
import threading
import random
from collections import deque
from pathlib import Path
from statistics import median

import cv2 as cv
import numpy as np
import ipywidgets as widgets
from IPython.display import display


# ==================== 集中配置 ====================

# --- 摄像头 ---
CAMERA_INDEX = None
FRAME_WIDTH  = 640
FRAME_HEIGHT = 480
CAMERA_FPS   = 30
FLIP_IMAGE   = True

# --- 机械臂（共用） ---
INITIAL_POSE       = [87, 70, 72, 36, 89, 30]
MOVE_TIME_MS       = 800
PAN_SERVO_ID       = 1
TILT_SERVO_ID      = 2
TILT_AUX_SERVO_ID  = 3
COLOR_SERVO_ID     = 4
PAN_LIMITS         = (10.0, 170.0)
TILT_LIMITS        = (30.0, 150.0)
TILT_AUX_LIMITS    = (20.0, 150.0)
COLOR_AUX_LIMITS   = (10.0, 120.0)
PAN_DIRECTION      = -1.0
TILT_DIRECTION     =  1.0

# --- PID / 死区 ---
PAN_DEAD_ZONE       = 16
TILT_DEAD_ZONE      = 28
CONTROL_INTERVAL    = 0.05
SERVO_MOVE_TIME_MS  = 500
SERVO_SKIP_CYCLES   = 2
PAN_KP              = 0.14
PAN_MAX_SPEED       = 24.0
PAN_ACCEL           = 200.0
TILT_KP             = 0.12
TILT_MAX_SPEED      = 20.0
TILT_ACCEL          = 160.0
TILT_HANDOFF_MARGIN = 15.0

# --- 速度平滑 ---
VELOCITY_SMOOTHING_ALPHA = 0.35

# --- 中心平滑 ---
PAN_SMOOTHING_ALPHA  = 0.75
TILT_SMOOTHING_ALPHA = 0.65
TARGET_MEDIAN_WINDOW = 1

# --- 人脸检测 ---
DETECTION_SCALE       = 0.5
MIN_FACE_SIZE         = (40, 40)
CASCADE_SCALE_FACTOR  = 1.2
CASCADE_MIN_NEIGHBORS = 5

# --- 颜色检测 ---
COLOR_BLOB_MIN_RADIUS = 10
COLOR_BLUR_KERNEL     = (5, 5)
COLOR_MORPH_ITER      = 2
COLOR_DEAD_ZONE       = 16

# --- 颜色 HSV 范围 ---
COLOR_HSV = {
    "red":    ((0, 25, 90),   (10, 255, 255)),
    "green":  ((53, 36, 40),  (80, 255, 255)),
    "blue":   ((110, 80, 90), (120, 255, 255)),
    "yellow": ((25, 20, 55),  (50, 255, 255)),
}
COLOR_NAMES = ["red", "green", "blue", "yellow"]

# --- 显示 ---
DISPLAY_EVERY_N_FRAMES = 2
JPEG_QUALITY           = 60

# --- 模式颜色 ---
STATUS_COLORS = {
    "General":      (180, 180, 180),
    "face_follow":  (0, 255, 0),
    "color_follow": (255, 100, 255),
}


# ==================== 自动探测 ====================

try:
    import Arm_Lib
    HAS_ARM = True
    print('[OK] Arm_Lib u5bfc入成功')
except ImportError:
    HAS_ARM = False
    print('[WARN] Arm_Lib u672a找到（非 200DK 环境）')

for cam_idx in range(3):
    cap_test = cv.VideoCapture(cam_idx)
    if cap_test.isOpened():
        CAMERA_INDEX = cam_idx
        cap_test.release()
        print(f'[OK] u6444像头已探测到，设备编号: {cam_idx}')
        break
if CAMERA_INDEX is None:
    print('[WARN] u672a探测到任何摄像头设备')

local_model = Path('haarcascade_frontalface_default.xml')
FACE_MODEL_PATH = (
    str(local_model) if local_model.exists()
    else cv.data.haarcascades + 'haarcascade_frontalface_default.xml'
)
print(f'[OK] u4eba脸模型: {FACE_MODEL_PATH}')

try:
    import keyboard
    HAS_KEYBOARD = True
    print('[OK] keyboard u5e93可用，u652f持全局快捷键')
except ImportError:
    HAS_KEYBOARD = False
    print('[WARN] keyboard u5e93未安装（pip install keyboard），u5c06使用按钮')


# Ctrl+Enter 运行本单元，确认配置无误后继续下一单元

[OK] Arm_Lib u5bfc入成功
[OK] u6444像头已探测到，设备编号: 0
[OK] u4eba脸模型: haarcascade_frontalface_default.xml
[WARN] keyboard u5e93未安装（pip install keyboard），u5c06使用按钮


## 2. 初始化机械臂

In [10]:
Arm = None
arm_lock = threading.Lock()


def write_arm_pose(pose, move_time_ms):
    if Arm is None:
        return
    angles = [int(round(v)) for v in pose]
    with arm_lock:
        if hasattr(Arm, 'Arm_serial_servo_write6_array'):
            Arm.Arm_serial_servo_write6_array(angles, int(move_time_ms))
        else:
            Arm.Arm_serial_servo_write6(*angles, int(move_time_ms))


if HAS_ARM:
    Arm = Arm_Lib.Arm_Device()
    time.sleep(0.2)
    write_arm_pose(INITIAL_POSE, MOVE_TIME_MS)
    time.sleep(MOVE_TIME_MS / 1000.0)
    print(f'[OK] u673a械臂已初始化到: {INITIAL_POSE}')
else:
    print('[SKIP] u8df3过机械臂初始化')

[OK] u673a械臂已初始化到: [87, 70, 72, 36, 89, 30]


## 3. 位置式 PID 控制器（时间制）

输出解释为角速度（度/秒），与人脸追踪方案一致。

In [11]:
class PositionPID:
    """时间制位置式 PID，u8f93出解释为角速度（度/秒）。"""

    def __init__(self, kp, ki=0.0, kd=0.0, output_limit=None):
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.output_limit = output_limit
        self.reset()

    def reset(self):
        self.integral = 0.0
        self.previous_error = 0.0
        self.previous_time = None

    def update(self, error, now):
        if self.previous_time is None:
            dt, derivative = 0.0, 0.0
        else:
            dt = max(now - self.previous_time, 1e-6)
            derivative = (error - self.previous_error) / dt
        self.integral += error * dt
        output = self.kp * error + self.ki * self.integral + self.kd * derivative
        if self.output_limit is not None:
            output = float(np.clip(output, -self.output_limit, self.output_limit))
        self.previous_error = error
        self.previous_time = now
        return output

## 4. 统一追踪类

**核心设计**：人脸追踪和颜色追踪共用同一套 P ID/**舵机平滑方案**（时间基速度 P ID + 低通滤波 + 加速度限幅 + 跳帧发送）。

**关节映射**：
- 人脸追踪：S1（平移）+ S2/S3（倾斜联动）
- 颜色追踪：S1（平移）+ S4（倾斜），S3 随 S4 联动一半


In [12]:
class UnifiedTracker:

    def __init__(self, arm=None):
        self.arm = arm
        self.face_cascade = cv.CascadeClassifier(FACE_MODEL_PATH)
        if self.face_cascade.empty():
            self.face_cascade = None
            print('[WARN] u4eba脸检测模型加载失败，u4eba脸模式不可用')

        # PID
        self.pan_pid  = PositionPID(PAN_KP,  output_limit=PAN_MAX_SPEED)
        self.tilt_pid = PositionPID(TILT_KP, output_limit=TILT_MAX_SPEED)

        # 角度
        self.pan_angle        = float(INITIAL_POSE[PAN_SERVO_ID - 1])
        self.tilt_angle       = float(INITIAL_POSE[TILT_SERVO_ID - 1])
        self.tilt_aux_angle   = float(INITIAL_POSE[TILT_AUX_SERVO_ID - 1])
        self.color_aux_angle  = float(INITIAL_POSE[COLOR_SERVO_ID - 1])

        # 速度
        self.pan_speed   = 0.0
        self.tilt_speed  = 0.0
        self.aux_speed   = 0.0

        # 速度低通滤波
        self.filtered_pan_speed  = 0.0
        self.filtered_tilt_speed = 0.0
        self.filtered_aux_speed  = 0.0

        # 控制时序
        self.last_control_time  = None
        self.last_send_time     = 0.0
        self._servo_cycle_count = 0

        # 中心平滑
        self.center_history  = deque(maxlen=TARGET_MEDIAN_WINDOW)
        self.smoothed_center = None

        self._mode = "General"

    # ---------- 工具 ----------
    @staticmethod
    def clamp(value, limits):
        return max(limits[0], min(value, limits[1]))

    @staticmethod
    def slew(current, target, max_change):
        return current + np.clip(target - current, -max_change, max_change)

    @staticmethod
    def face_filter(faces):
        valid = [(int(x), int(y), int(w), int(h))
                 for x, y, w, h in faces
                 if w >= 10 and h >= 10]
        return max(valid, key=lambda f: f[2] * f[3]) if valid else None

    def reset_pid(self):
        self.pan_pid.reset()
        self.tilt_pid.reset()
        self.pan_speed   = 0.0
        self.tilt_speed  = 0.0
        self.aux_speed   = 0.0
        self.filtered_pan_speed   = 0.0
        self.filtered_tilt_speed  = 0.0
        self.filtered_aux_speed   = 0.0
        self.last_control_time = None
        self._servo_cycle_count = 0
        self.center_history.clear()
        self.smoothed_center = None

    def set_mode(self, new_mode):
        if new_mode != self._mode:
            self._mode = new_mode
            self.reset_pid()
            d = {"General":"待机", "face_follow":"人脸追踪", "color_follow":"颜色追踪"}
            print(f'[MODE] u5207换到: {d.get(new_mode, new_mode)}')

    @property
    def mode(self):
        return self._mode

    # ---------- 人脸追踪舵机控制 (S1+S2/S3联动) ----------
    def _update_servos_face(self, cx, cy):
        now = time.monotonic()
        if now - self.last_send_time < CONTROL_INTERVAL:
            return
        dt = (CONTROL_INTERVAL if self.last_control_time is None
              else min(now - self.last_control_time, 0.1))
        self.last_control_time = now
        self.last_send_time    = now

        ex = FRAME_WIDTH / 2.0 - cx
        ey = FRAME_HEIGHT / 2.0 - cy
        if abs(ex) < PAN_DEAD_ZONE:  ex = 0.0
        if abs(ey) < TILT_DEAD_ZONE: ey = 0.0

        tps = 0.0 if ex == 0.0 else PAN_DIRECTION  * self.pan_pid.update(ex, now)
        tts = 0.0 if ey == 0.0 else TILT_DIRECTION * self.tilt_pid.update(ey, now)

        a = VELOCITY_SMOOTHING_ALPHA
        self.filtered_pan_speed  = a * self.filtered_pan_speed  + (1.0 - a) * tps
        self.filtered_tilt_speed = a * self.filtered_tilt_speed + (1.0 - a) * tts

        self.pan_speed  = 0.0 if ex == 0.0 else self.slew(self.pan_speed,  self.filtered_pan_speed,  PAN_ACCEL  * dt)
        self.tilt_speed = 0.0 if ey == 0.0 else self.slew(self.tilt_speed, self.filtered_tilt_speed, TILT_ACCEL * dt)

        pan_d  = self.pan_speed  * dt
        tilt_d = self.tilt_speed * dt
        self.pan_angle = self.clamp(self.pan_angle + pan_d, PAN_LIMITS)

        dist = ((TILT_AUX_LIMITS[1] - self.tilt_aux_angle) if tilt_d >= 0
                else (self.tilt_aux_angle - TILT_AUX_LIMITS[0]))
        share = float(np.clip(dist / TILT_HANDOFF_MARGIN, 0.0, 1.0))
        old_a = self.tilt_aux_angle
        self.tilt_aux_angle = self.clamp(self.tilt_aux_angle + tilt_d * share, TILT_AUX_LIMITS)
        rem = tilt_d - (self.tilt_aux_angle - old_a)
        self.tilt_angle = self.clamp(self.tilt_angle + rem, TILT_LIMITS)

        self._servo_cycle_count += 1
        if self._servo_cycle_count % SERVO_SKIP_CYCLES == 0:
            pose = list(INITIAL_POSE)
            pose[PAN_SERVO_ID - 1]      = self.pan_angle
            pose[TILT_SERVO_ID - 1]     = self.tilt_angle
            pose[TILT_AUX_SERVO_ID - 1] = self.tilt_aux_angle
            if self.arm is not None:
                write_arm_pose(pose, SERVO_MOVE_TIME_MS)

    # ---------- 颜色追踪舵机控制 (S1+S4) ----------
    def _update_servos_color(self, cx, cy):
        now = time.monotonic()
        if now - self.last_send_time < CONTROL_INTERVAL:
            return
        dt = (CONTROL_INTERVAL if self.last_control_time is None
              else min(now - self.last_control_time, 0.1))
        self.last_control_time = now
        self.last_send_time    = now

        ex = FRAME_WIDTH / 2.0 - cx
        ey = FRAME_HEIGHT / 2.0 - cy
        if abs(ex) < COLOR_DEAD_ZONE: ex = 0.0
        if abs(ey) < COLOR_DEAD_ZONE: ey = 0.0

        tps = 0.0 if ex == 0.0 else PAN_DIRECTION  * self.pan_pid.update(ex, now)
        tas = 0.0 if ey == 0.0 else TILT_DIRECTION * self.tilt_pid.update(ey, now)

        a = VELOCITY_SMOOTHING_ALPHA
        self.filtered_pan_speed = a * self.filtered_pan_speed + (1.0 - a) * tps
        self.filtered_aux_speed = a * self.filtered_aux_speed + (1.0 - a) * tas

        self.pan_speed = 0.0 if ex == 0.0 else self.slew(self.pan_speed, self.filtered_pan_speed, PAN_ACCEL * dt)
        self.aux_speed = 0.0 if ey == 0.0 else self.slew(self.aux_speed, self.filtered_aux_speed, PAN_ACCEL * dt)

        self.pan_angle       = self.clamp(self.pan_angle + self.pan_speed * dt, PAN_LIMITS)
        self.color_aux_angle = self.clamp(self.color_aux_angle + self.aux_speed * dt, COLOR_AUX_LIMITS)

        self._servo_cycle_count += 1
        if self._servo_cycle_count % SERVO_SKIP_CYCLES == 0:
            pose = list(INITIAL_POSE)
            pose[PAN_SERVO_ID - 1]      = self.pan_angle
            pose[COLOR_SERVO_ID - 1]    = self.color_aux_angle
            pose[TILT_SERVO_ID - 1]     = self.tilt_angle
            pose[TILT_AUX_SERVO_ID - 1] = self.color_aux_angle / 2.0
            if self.arm is not None:
                write_arm_pose(pose, SERVO_MOVE_TIME_MS)

    # ---------- 人脸检测 ----------
    def face_detect(self, frame):
        if self.face_cascade is None:
            return None
        df = cv.resize(frame, None, fx=DETECTION_SCALE, fy=DETECTION_SCALE, interpolation=cv.INTER_AREA)
        g = cv.cvtColor(df, cv.COLOR_BGR2GRAY)
        g = cv.equalizeHist(g)
        ms = (max(10, int(MIN_FACE_SIZE[0] * DETECTION_SCALE)),
              max(10, int(MIN_FACE_SIZE[1] * DETECTION_SCALE)))
        sf = self.face_cascade.detectMultiScale(g, scaleFactor=CASCADE_SCALE_FACTOR,
                                                  minNeighbors=CASCADE_MIN_NEIGHBORS, minSize=ms)
        inv = 1.0 / DETECTION_SCALE
        faces = [tuple(int(round(v * inv)) for v in f) for f in sf]
        return self.face_filter(faces)

    # ---------- 颜色检测 ----------
    def color_detect(self, frame, hsv_cfg):
        lo, hi = hsv_cfg
        hsv = cv.cvtColor(frame, cv.COLOR_BGR2HSV)
        mask = cv.inRange(hsv, lo, hi)
        mask = cv.erode(mask, None, iterations=COLOR_MORPH_ITER)
        mask = cv.dilate(mask, None, iterations=COLOR_MORPH_ITER)
        mask = cv.GaussianBlur(mask, COLOR_BLUR_KERNEL, 0)
        cnts = cv.findContours(mask.copy(), cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)[-2]
        if not cnts:
            return None
        cnt = max(cnts, key=cv.contourArea)
        (cx, cy), r = cv.minEnclosingCircle(cnt)
        if r < COLOR_BLOB_MIN_RADIUS:
            return None
        return (cx, cy, r)

    # ---------- 中心平滑 ----------
    def smooth_center(self, rc):
        if rc is None:
            self.smoothed_center = None
            return None
        self.center_history.append(rc)
        mc = (float(median(p[0] for p in self.center_history)),
              float(median(p[1] for p in self.center_history)))
        if self.smoothed_center is None:
            self.smoothed_center = mc
        else:
            self.smoothed_center = (
                PAN_SMOOTHING_ALPHA  * mc[0] + (1.0 - PAN_SMOOTHING_ALPHA)  * self.smoothed_center[0],
                TILT_SMOOTHING_ALPHA * mc[1] + (1.0 - TILT_SMOOTHING_ALPHA) * self.smoothed_center[1],
            )
        return (int(round(self.smoothed_center[0])), int(round(self.smoothed_center[1])))

    # ---------- 主追踪 ----------
    def track(self, frame, color_name="red"):
        frame = cv.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
        info = {"mode": self._mode, "center": None, "color": None}

        if self._mode == "face_follow":
            face = self.face_detect(frame)
            if face is None:
                self.reset_pid()
                cv.putText(frame, 'NO FACE', (10, 32), cv.FONT_HERSHEY_DUPLEX, 0.7, (0, 0, 255), 2)
                cv.drawMarker(frame, (FRAME_WIDTH//2, FRAME_HEIGHT//2), (255,255,255), cv.MARKER_CROSS, 24, 2)
                return frame, False, info
            x, y, w, h = face
            center = self.smooth_center((x + w//2, y + h//2))
            if center is None:
                return frame, False, info
            self._update_servos_face(center[0], center[1])
            cv.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv.circle(frame, center, 5, (0, 0, 255), -1)
            cv.drawMarker(frame, (FRAME_WIDTH//2, FRAME_HEIGHT//2), (255,255,255), cv.MARKER_CROSS, 24, 2)
            cv.line(frame, (FRAME_WIDTH//2, FRAME_HEIGHT//2), center, (255, 255, 0), 2)
            info["center"] = center
            return frame, True, info

        elif self._mode == "color_follow":
            hc = COLOR_HSV.get(color_name, COLOR_HSV["red"])
            r = self.color_detect(frame, hc)
            if r is None:
                self.reset_pid()
                cv.putText(frame, f'NO {color_name.upper()}', (10, 32), cv.FONT_HERSHEY_DUPLEX, 0.7, (0,0,255), 2)
                cv.drawMarker(frame, (FRAME_WIDTH//2, FRAME_HEIGHT//2), (255,255,255), cv.MARKER_CROSS, 24, 2)
                return frame, False, info
            cx, cy, radius = r
            center = self.smooth_center((cx, cy))
            if center is None:
                return frame, False, info
            self._update_servos_color(center[0], center[1])
            cv.circle(frame, (int(cx), int(cy)), int(radius), (255, 0, 255), 2)
            cv.circle(frame, center, 5, (0, 0, 255), -1)
            cv.drawMarker(frame, (FRAME_WIDTH//2, FRAME_HEIGHT//2), (255,255,255), cv.MARKER_CROSS, 24, 2)
            cv.line(frame, (FRAME_WIDTH//2, FRAME_HEIGHT//2), center, (255, 255, 0), 2)
            cv.putText(frame, color_name.upper(), (center[0]-30, center[1]-22),
                       cv.FONT_HERSHEY_DUPLEX, 0.6, (0,255,255), 2)
            info["center"] = center
            info["color"] = color_name
            return frame, True, info

        else:
            cv.putText(frame, 'STANDBY - Press key to start', (90, FRAME_HEIGHT // 2),
                       cv.FONT_HERSHEY_DUPLEX, 0.65, (180,180,180), 2)
            cv.drawMarker(frame, (FRAME_WIDTH//2, FRAME_HEIGHT//2), (255,255,255), cv.MARKER_CROSS, 24, 2)
            return frame, False, info

## 5. 创建实例与全局状态

初始化追踪器、模式变量、FPS 计数器。

In [13]:
tracker = UnifiedTracker(Arm)
current_color = "red"
model = "General"

_fps_last_time   = time.time()
_fps_frame_count = 0
_fps_display     = 0.0

## 6. 键盘监听线程

使用 `keyboard` 库在后台监听全局快捷键。

| 按键 | 模式 |
|------|------|
| **F** | 人脸追踪 |
| **C** | 颜色追踪 |
| **1/2/3/4** | 选红/绿/蓝/黄 + 切换到颜色追踪 |
| **G** | 取消追踪 |
| **Q / ESC** | 退出 |

如果 `keyboard` 库不可用，使用 ipywidgets 按钮操作。

In [14]:
_keyboard_thread_stop = threading.Event()

def keyboard_listener():
    global model, current_color
    if not HAS_KEYBOARD:
        return
    key_map = {'f': 'face_follow', 'c': 'color_follow', 'g': 'General', 'q': 'Exit', 'esc': 'Exit'}
    while not _keyboard_thread_stop.is_set():
        try:
            event = keyboard.read_event()
            if event.event_type == keyboard.KEY_DOWN:
                key = event.name.lower()
                if key in key_map:
                    model = key_map[key]
                elif key in ('1', '2', '3', '4'):
                    idx = int(key) - 1
                    current_color = COLOR_NAMES[idx]
                    model = 'color_follow'
                    print(f'[KEY] u5207换到颜色追踪: {current_color}')
            if model == 'Exit':
                break
        except Exception:
            time.sleep(0.1)
    print('[KEY] u952e盘监听线程已退出')

def start_keyboard_thread():
    if HAS_KEYBOARD:
        _keyboard_thread_stop.clear()
        t = threading.Thread(target=keyboard_listener, daemon=True)
        t.start()
        print('[KEY] u5168局快捷键已启用: f=人脸 c=颜色 1-4=选色 g=取消 q=退出')

def stop_keyboard_thread():
    _keyboard_thread_stop.set()

## 7. 创建控件

键盘快捷键为主，按钮为备用方案。同时支持两种操作方式。

In [15]:
button_layout = widgets.Layout(width='180px', height='70px', align_self='center')
output = widgets.Output()

# 模式按钮
face_btn = widgets.Button(description='[F] u4eba脸追踪', button_style='info', layout=button_layout)
color_btn = widgets.Button(description='[C] u989c色追踪', button_style='success', layout=button_layout)
cancel_btn = widgets.Button(description='[G] u53d6消追踪', button_style='warning', layout=button_layout)
exit_btn = widgets.Button(description='[Q] u9000出', button_style='danger', layout=button_layout)

# 颜色选择
choose_color = widgets.ToggleButtons(
    options=COLOR_NAMES, value='red', button_style='success',
    tooltips=['u7ea2色 [1]', 'u7eff色 [2]', 'u84dd色 [3]', 'u9ec4色 [4]'],
    layout=widgets.Layout(width='auto'),
)

# 图像
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='auto'))

# 状态标签
status_label = widgets.Label(value='模式: 待机  |  颜色: red', layout=widgets.Layout(margin='4px 0'),
                            style={'font_size': '13px', 'font_weight': 'bold'})


def face_btn_cb(_):  global model; model = 'face_follow'
def color_btn_cb(_): global model; model = 'color_follow'
def cancel_btn_cb(_): global model; model = 'General'
def exit_btn_cb(_):  global model; model = 'Exit'

face_btn.on_click(face_btn_cb)
color_btn.on_click(color_btn_cb)
cancel_btn.on_click(cancel_btn_cb)
exit_btn.on_click(exit_btn_cb)

# 布局
main_vbox = widgets.VBox([
    status_label,
    widgets.HBox([
        widgets.VBox([imgbox, choose_color]),
        widgets.VBox([face_btn, color_btn, cancel_btn, exit_btn]),
    ])
])

display(main_vbox, output)

Output()

[MODE] u5207换到: 颜色追踪
[MODE] u5207换到: 待机
[MODE] u5207换到: Exit


## 8. 摄像头主循环 + 启动

运行此单元启动追踪系统。按 **F** 开始人脸追踪，按 **C** 开始颜色追踪。

In [16]:
def camera():
    global model, current_color, _fps_last_time, _fps_frame_count, _fps_display

    capture = cv.VideoCapture(CAMERA_INDEX)
    if not capture.isOpened():
        with output:
            print(f'[ERROR] 无法打开摄像头 /dev/video{CAMERA_INDEX}')
        return

    capture.set(3, FRAME_WIDTH)
    capture.set(4, FRAME_HEIGHT)
    capture.set(5, CAMERA_FPS)
    capture.set(cv.CAP_PROP_BUFFERSIZE, 1)

    with output:
        print(f'[OK] 摄像头已打开（/dev/video{CAMERA_INDEX}）')
        print('=' * 50)
        print('  按键指南：')
        print('    F 键 → 人脸追踪')
        print('    C 键 → 颜色追踪')
        print('    1-4  → 选择颜色（红/绿/蓝/黄）')
        print('    G 键 → 取消追踪（待机）')
        print('    Q/ESC → 退出')
        print('=' * 50)

    last_mode = "General"
    frame_count = 0

    while capture.isOpened() and model != 'Exit':
        try:
            ret, img = capture.read()
            if not ret:
                time.sleep(0.005)
                continue

            if FLIP_IMAGE:
                img = cv.flip(img, 1)

            # 模式同步
            if model != last_mode:
                tracker.set_mode(model)
                last_mode = model
            if model == 'color_follow' and choose_color.value:
                current_color = choose_color.value

            # 追踪
            img, has_target, info = tracker.track(img, current_color)

            # ---- 顶部半透明背景条 ----
            overlay = img.copy()
            cv.rectangle(overlay, (0, 0), (FRAME_WIDTH, 36), (30, 30, 30), -1)
            cv.addWeighted(overlay, 0.45, img, 0.55, 0, img)

            bar_color = STATUS_COLORS.get(model, (180, 180, 180))
            cv.rectangle(img, (0, 0), (FRAME_WIDTH, 3), bar_color, -1)

            mt = {"General": "STANDBY", "face_follow": "FACE TRACKING",
                  "color_follow": f"COLOR TRACKING [{current_color.upper()}]"}
            cv.putText(img, mt.get(model, model), (10, 26),
                       cv.FONT_HERSHEY_DUPLEX, 0.55, bar_color, 1)

            # FPS (右对齐)
            _fps_frame_count += 1
            now = time.time()
            if now - _fps_last_time >= 1.0:
                _fps_display = _fps_frame_count / (now - _fps_last_time)
                _fps_frame_count = 0
                _fps_last_time = now
            fps_text = f'FPS: {_fps_display:.1f}'
            (fw, fh), _ = cv.getTextSize(fps_text, cv.FONT_HERSHEY_DUPLEX, 0.5, 1)
            cv.putText(img, fps_text, (FRAME_WIDTH - fw - 10, 26),
                       cv.FONT_HERSHEY_DUPLEX, 0.5, (0, 255, 255), 1)

            # ---- 底部半透明状态栏 ----
            overlay2 = img.copy()
            cv.rectangle(overlay2, (0, FRAME_HEIGHT - 46), (FRAME_WIDTH, FRAME_HEIGHT), (30, 30, 30), -1)
            cv.addWeighted(overlay2, 0.45, img, 0.55, 0, img)
            cv.line(img, (0, FRAME_HEIGHT - 46), (FRAME_WIDTH, FRAME_HEIGHT - 46), (60, 60, 60), 1)

            # 舵机角度（第一行）
            if model == 'face_follow':
                cv.putText(img, f'S1:{tracker.pan_angle:.0f}  S2:{tracker.tilt_angle:.0f}  S3:{tracker.tilt_aux_angle:.0f}',
                           (10, FRAME_HEIGHT - 28), cv.FONT_HERSHEY_DUPLEX, 0.45, (0, 255, 0), 1)
            elif model == 'color_follow':
                cv.putText(img, f'S1:{tracker.pan_angle:.0f}  S4:{tracker.color_aux_angle:.0f}',
                           (10, FRAME_HEIGHT - 28), cv.FONT_HERSHEY_DUPLEX, 0.45, (255, 100, 255), 1)

            # 键盘提示（第二行）
            cv.putText(img, '[F]ace  [C]olor  [1-4]Color  [G]eneral  [Q]uit',
                       (10, FRAME_HEIGHT - 8), cv.FONT_HERSHEY_DUPLEX, 0.42, (160, 160, 160), 1)

            # 编码显示
            if frame_count % DISPLAY_EVERY_N_FRAMES == 0:
                ret, jpg = cv.imencode('.jpg', img, [cv.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
                if ret:
                    imgbox.value = jpg.tobytes()
            frame_count += 1

            status_label.value = f'模式: {mt.get(model, model)}  |  FPS: {_fps_display:.1f}'

        except KeyboardInterrupt:
            break
        except Exception as e:
            with output:
                print(f'[WARN] {type(e).__name__}: {e}')
            time.sleep(0.1)

    capture.release()
    cv.destroyAllWindows()
    stop_keyboard_thread()
    with output:
        print('[INFO] 程序已退出，摄像头资源已释放')


if CAMERA_INDEX is not None:
    start_keyboard_thread()
    threading.Thread(target=camera, daemon=True).start()
    print('[INFO] 摄像头线程已启动')
else:
    with output:
        print('[ERROR] 未探测到摄像头设备，程序无法启动！')


[INFO] 摄像头线程已启动


## 调参建议

### 通用
- **水平/竖直方向相反**：反转 `PAN_DIRECTION` / `TILT_DIRECTION` 的正负号
- **运动顿挫**：增大 `SERVO_MOVE_TIME_MS`（300→500），增大 `VELOCITY_SMOOTHING_ALPHA`（0.35→0.5）
- **响应太慢**：减小 `SERVO_SKIP_CYCLES`（2→1），增大 `PAN_KP` / `TILT_KP`
- **中心附近抖动**：增大 `PAN_DEAD_ZONE` / `TILT_DEAD_ZONE`

### 人脸追踪
- **检测误报/漏检**：调整 `CASCADE_SCALE_FACTOR` / `CASCADE_MIN_NEIGHBORS`
- **跟踪不平滑**：调整 `PAN_SMOOTHING_ALPHA` / `TILT_SMOOTHING_ALPHA`

### 颜色追踪
- **色块检测不到**：调整 `COLOR_HSV` 中的 HSV 范围
- **小色块抖动**：增大 `COLOR_BLOB_MIN_RADIUS`

每次只调整一个参数。